# Temporal Hawkes processes

This notebook walks through the three purely temporal process classes. It is
executed on every documentation build, so everything below is guaranteed to run
against the released code.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import hawkes_package as hp

# Exponential Hawkes process

The first simulation we are going to look at is the most simple but also most prominent type of Hawkes processes, the exponential type Hawkes process. In this case, the kernel is defined via an exponential function, i.e.,
$$h(x) = \alpha\mathrm{exp}(-\beta x)$$.
The class `ExponentialHawkes` takes an array of length 3, `[mu, alpha, beta]`: the background rate, the excitation size and the decay rate. Recall that $\frac{\alpha}{\beta}<1$ should be satisfied to avoid explosion
(the simulation will in any case give you a fitting result, so you may "see" explosion when choosing parameters outside this bound). 

## Ogata's Thinning Algorithm

All simulations in this package use **Ogata's thinning (Lewis–Shedler) method**. The idea is to dominate the true conditional intensity $\lambda^*(t)$ with a piecewise-constant upper bound $M(t)$, generate candidate event times from a Poisson process at rate $M$, and accept each candidate with probability $\lambda^*(t)/M$.

```
1. Start at the current time t₀ (last accepted event).
2. Compute an upper bound  M ≥ λ*(s)  for all s ≥ t₀.
3. Draw  U ~ Uniform(0,1),  set  τ = −log(U)/M,  advance  t = t₀ + τ.
4. Draw  V ~ Uniform(0,1).
5. If  V ≤ λ*(t)/M  → accept: record event at t, set t₀ = t, go to 2.
   Else              → reject:              go to 2 (reuse current t as new t₀).
```

**Correctness requirement:** $M$ must satisfy $M \geq \lambda^*(s)$ for **all** $s \geq t_0$, including the new $t_0 = t_{\text{last accepted}}$. Failing this makes the acceptance ratio exceed 1 and collapses the algorithm to a simple Poisson process.

The package computes $M$ by bounding **each past event's future contribution
separately** — by the kernel's peak value if that event has not yet peaked, and
by its current value if it has. A bound that is too *tight* is the dangerous
failure mode: it does not raise, it silently degrades the process towards
Poisson.

## Stability conditions

A Hawkes process is **stable** (does not explode to infinitely many events in finite time) when the expected number of offspring per event is less than 1.

| Class | Stability condition |
|---|---|
| `ExponentialHawkes([μ, α, β])` | α/β < 1 |
| `MonotoneKernelHawkes(κ_t, g)` | depends on kernel norm and g |
| `BellShapeHawkes(κ_t, g)` | spectral radius of branching matrix < 1 |

The package enforces the exponential condition at construction time and raises `ValueError` if violated.

In [ ]:
# alpha/beta = 5 >= 1 is not stationary, so the constructor refuses to build a
# process whose simulation would never terminate.
try:
    hp.ExponentialHawkes(np.array([1.0, 5.0, 1.0]))
except ValueError as e:
    print(f"Caught expected error:\n  {e}")

In [ ]:
# [mu=2, alpha=0.5, beta=1]: branching ratio alpha/beta = 0.5 < 1, so this is stable.
# `rng=` makes the run reproducible. As of 0.2.0 `np.random.seed` has no effect.
G = hp.ExponentialHawkes(np.array([2.0, 0.5, 1.0]), rng=42)
G.simulate(100)
print(f"{len(G.Events)} events, last at t = {G.Events[-1]:.2f}")

In [ ]:
x_G = np.linspace(0, G.Events[-1], 1000)
y_G, z_G = G.intensity_over_interval(x_G)

plt.figure(figsize=(9, 3))
plt.plot(y_G, z_G)
plt.axhline(2.0, ls="--", c="grey", lw=1, label=r"background $\mu$")
plt.xlabel("time")
plt.ylabel(r"$\lambda(t \mid H_t)$")
plt.legend()
plt.tight_layout()
plt.show()

The curve never drops below the dashed background rate $\mu$. Before 0.2.0 the
accessor omitted $\mu$, so this plot sat a constant $2.0$ too low — the
simulator and the accessor disagreed about what the intensity was.

# Monotone Hawkes process

A widely used class of Hawkes processes are characterized by their monotonously decreasing kernels. This can be seen as an inital even that instantanously increases the intensity but whichs impact then faints over time. The exponential case is a spacial case of the monotone Hawkes process. The THP class furthermore allows nonlinearities that are monotonously increasing and defined on $[0,\infty)$. Just give an optional input `nonlinearity`. 
You may, thus, check stability numerically for your process to get a first intuition about its behaviour.

In [ ]:
H = hp.MonotoneKernelHawkes(
    lambda x: 0.4 * np.exp(-10 * x), nonlinearity=lambda x: np.exp(x), rng=7
)
H.simulate(500)
print(f"{len(H.Events)} events over t = 0 .. {H.Events[-1]:.1f}")

In [ ]:
x = np.linspace(0, H.Events[-1], 100)
y, z = H.intensity_over_interval(x)

plt.figure(figsize=(9, 3))
plt.plot(y, z)
plt.xlabel("time")
plt.ylabel(r"$\lambda(t \mid H_t)$")
plt.tight_layout()
plt.show()

## Nonlinearity and stability

The kernel amplitude above is $0.4$, not $1$, and that matters. With
$\varphi = \exp$ each event multiplies the intensity by $e^{\kappa(0)}$, so a
unit-amplitude kernel makes the expected offspring count exceed one and the
process explodes: the intensity diverges, inter-arrival times underflow to
zero, and time stops advancing.

The package detects that and raises rather than looping forever.

In [ ]:
explosive = hp.MonotoneKernelHawkes(
    lambda x: 1.0 * np.exp(-10 * x), nonlinearity=lambda x: np.exp(x), rng=7
)
try:
    explosive.simulate(500)
except RuntimeError as e:
    print(f"Caught expected error:\n  {e}")

# Hawkes processes with bell shaped kernels

Hawkes processes with in the widest sense bell shaped kernels, for example triangular kernels, which have exactly one extremum, being a global maximum, give a modeling approach for intensities that do not jump immediately as soon as an event takes place but rather let it grow over time before decaying again. 
This THP class also allows monotonously increasing nonlinearities.

In [ ]:
def triangular_kernel(x):
    growing = (x > 0) & (x < 1 / 2)
    decaying = (x > 1 / 2) & (x < 1)
    return 2 * x * growing + (-2 * x + 2) * decaying


K = hp.BellShapeHawkes(triangular_kernel, rng=3)
print(f"kernel peaks at lag {K.ext:.3f}")

## Kernel comparison

The choice of temporal kernel shapes which type of self-excitation the process models:

- **Exponential** $\alpha e^{-\beta t}$: immediate jump at the event, then exponential decay. Memoryless; tractable likelihood.
- **Monotone (general)**: any decreasing kernel, e.g. power-law $t^{-p}$. Can model heavier-tailed clustering.
- **Bell-shaped (triangular)**: intensity rises *after* each event before decaying. Models delayed reactions or signal propagation.

In [ ]:
tau = np.linspace(0.01, 2, 300)

alpha, beta = 0.9, 2.0
exponential = alpha * np.exp(-beta * tau)
monotone = 0.9 * np.exp(-10 * tau)
bell = np.vectorize(triangular_kernel)(tau)

plt.figure(figsize=(8, 3))
plt.plot(tau, exponential, label=r"Exponential  $\alpha e^{-\beta t}$")
plt.plot(tau, monotone, label=r"Monotone  $e^{-10t}$")
plt.plot(tau, bell, label="Bell-shaped (triangular)")
plt.xlabel("lag $t$")
plt.ylabel("kernel value")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
K.simulate(50)

x_K = np.linspace(0, K.Events[-1], 1000)
y_K, z_K = K.intensity_over_interval(x_K)

plt.figure(figsize=(9, 3))
plt.plot(y_K, z_K)
plt.xlabel("time")
plt.ylabel(r"$\lambda(t \mid H_t)$")
plt.tight_layout()
plt.show()

Notice the intensity climbing *after* each event rather than jumping at it —
the defining feature of a bell-shaped kernel.